# 基于Spikingjelly实现SNN完成MNIST分类

In [2]:
import torch
import torch.nn as nn
from spikingjelly.activation_based import neuron, functional, surrogate, layer, encoding
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## 定义SNN模型

In [3]:
# 定义简单 CNN 脉冲神经网络
class CSNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.T = 20 # 时间步长
        # 频率编码器：将像素值转换为泊松脉冲序列
        self.encoder = encoding.PoissonEncoder()
        
        # 网络结构：Conv -> IFNode -> MaxPool -> Conv -> IFNode -> MaxPool -> Flatten -> Linear -> IFNode
        self.static_conv = nn.Sequential(
            layer.Conv2d(1, 16, kernel_size=3, padding=1, bias=False),
            layer.BatchNorm2d(16),
            neuron.IFNode(surrogate_function=surrogate.ATan()), # 代理梯度用于反向传播
            layer.MaxPool2d(2, 2),
            
            layer.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
            layer.BatchNorm2d(32),
            neuron.IFNode(surrogate_function=surrogate.ATan()),
            layer.MaxPool2d(2, 2)
        )
        
        self.fc = nn.Sequential(
            layer.Flatten(),
            layer.Linear(32 * 7 * 7, 10, bias=True),
            neuron.IFNode(surrogate_function=surrogate.ATan())
        )

    def forward(self, x):
        # 初始化/清空神经元膜电位状态
        functional.reset_net(self)
        
        out_spikes_counter = 0
        for t in range(self.T):
            # 频率编码：每个时间步将图像转为脉冲
            x_spk = self.encoder(x)
            # 经过卷积层和全连接层
            x_out = self.static_conv(x_spk)
            x_out = self.fc(x_out)
            # 累加输出层的脉冲数用于最后分类
            out_spikes_counter += x_out
            
        # 计算平均发放频率作为预测依据
        return out_spikes_counter / self.T

In [4]:
# 1. 准备 MNIST 数据集
batch_size = 64
train_dataset = datasets.MNIST(root='./MNIST', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='./MNIST', train=False, transform=transforms.ToTensor(), download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [5]:
# 2. 实例化模型、定义损失函数和优化器
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CSNN().to(device)
criterion = nn.MSELoss() # SNN常用的频率分类损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [7]:
# 3. 训练模型
epochs = 5
print(f"[Device: {device}] Starting training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_acc = 0
    train_samples = 0
    
    for batch_idx, (img, label) in enumerate(train_loader):
        img, label = img.to(device), label.to(device)
        label_onehot = torch.nn.functional.one_hot(label, 10).float()
        
        optimizer.zero_grad()
        # out_fr 的形状: [batch_size, 10]
        out_fr = model(img)  
        loss = criterion(out_fr, label_onehot)
        loss.backward()
        optimizer.step()
        
        # 统计平均损失
        train_loss += loss.item() * label.numel()
        
        # 统计准确率：找到发放频率最高的神经元
        # out_fr.argmax(1) 会返回预测的数字标签
        train_acc += (out_fr.argmax(1) == label).float().sum().item()
        train_samples += label.numel()

        if (batch_idx + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}, Accuracy: {100.0 * train_acc / train_samples:.2f}%')
        
    # 计算该 Epoch 的平均值
    avg_train_loss = train_loss / train_samples
    avg_train_acc = train_acc / train_samples

    # 打印结果
    print(f"Epoch {epoch + 1}: Train Loss = {avg_train_loss:.4f}, Train Acc = {avg_train_acc * 100:.2f}%")

    # --- 可选：在每个 Epoch 结束后运行一次测试集验证 ---
    model.eval()
    test_acc = 0
    test_samples = 0
    with torch.no_grad():
        for img, label in test_loader:
            img, label = img.to(device), label.to(device)
            out_fr = model(img)
            test_acc += (out_fr.argmax(1) == label).float().sum().item()
            test_samples += label.numel()
    print(f"         Test Acc  = {(test_acc / test_samples) * 100:.2f}%")

[Device: cuda] Starting training...
Epoch [1/5], Batch [100/938], Loss: 0.0055, Accuracy: 97.44%
Epoch [1/5], Batch [200/938], Loss: 0.0073, Accuracy: 97.41%
Epoch [1/5], Batch [300/938], Loss: 0.0110, Accuracy: 97.49%
Epoch [1/5], Batch [400/938], Loss: 0.0022, Accuracy: 97.61%
Epoch [1/5], Batch [500/938], Loss: 0.0036, Accuracy: 97.67%
Epoch [1/5], Batch [600/938], Loss: 0.0030, Accuracy: 97.70%


KeyboardInterrupt: 